# 25｜葡萄牙污水冻结后身份揭盲：已知污染物恢复与排序富集评价

本笔记本只在24号身份盲态排序已经冻结并通过哈希核验后，才打开化合物身份。它把论文补充材料中的107个唯一CEC（原表108行，Losartan重复）连接到公共**实测**MS/MS参考谱，对真实污水DDA特征执行严格谱库匹配，并评价冻结PBT危险性排序、出现度辅助排序、丰度基线和检出频率基线。

核心边界：未匹配特征是**未标注**，不是阴性；无保留时间标准，因此非歧义谱库命中最多报告为Schymanski Level 2a，绝不宣称Level 1；峰面积和检出频率不是浓度，因此这里仍是危险性/优先级排序，不是定量风险评估。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
import pandas as pd
from IPython.display import display, Image

ROOT = Path(r'E:\谱图预测PBT_终版')
ENV_MS = Path(r'D:\Appinstallation\Anaconda\envs\env_ms\python.exe')
SRC = ROOT / 'src'
CASE = ROOT / 'external_data' / 'portugal_wastewater_5830725'
INPUTS = CASE / '06_revealed_evaluation' / 'inputs'
OUT = CASE / '06_revealed_evaluation' / '25_identity_reveal'
MANIFEST_DIR = CASE / 'manifests'

# 正常首次运行保持三个开关均为 True；如某阶段已成功且只想续跑，可把前面的阶段设为 False。
RUN_PREPARE = True
RUN_MATCH = True
RUN_EVALUATE = True

assert ROOT.is_dir(), ROOT
assert ENV_MS.is_file(), ENV_MS
assert SRC.is_dir(), SRC
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + os.pathsep + env.get('PYTHONPATH', '')
env['PYTHONIOENCODING'] = 'utf-8'

def run_stage(stage):
    cmd = [str(ENV_MS), '-m', 'pbt_msms.portugal_identity_reveal',
           '--stage', stage, '--root', str(ROOT)]
    print('Running:', ' '.join(cmd))
    started = time.perf_counter()
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)
    print(f'{stage} finished in {(time.perf_counter() - started) / 60:.2f} min')

print('env_ms:', ENV_MS)
print('output:', OUT)

## 1. 24号冻结门控与身份来源审计

这一步只核对前序状态和输入来源，不做匹配。必须满足：24号状态为 `READY_FOR_NOTEBOOK_25_IDENTITY_REVEAL`、排序明确为身份盲态、冻结文件存在。论文目标表来自补充材料Table SM-10；结构桥来自Table SM-3与PubChem快照；8个Watch‑List CEC来自原论文正文。模块在真正计算前还会逐个核对24号输出哈希。

In [ ]:
run24_path = MANIFEST_DIR / '24_hazard_ranking_run_manifest.json'
freeze24_path = MANIFEST_DIR / '24_identity_blind_freeze.json'
supplement_manifest_path = INPUTS / '25_supplement_extraction_manifest.json'
pubchem_manifest_path = INPUTS / '25_pubchem_identity_manifest.json'
watchlist_path = INPUTS / '25_published_watchlist_cecs.csv'

for path in [run24_path, freeze24_path, supplement_manifest_path,
             pubchem_manifest_path, watchlist_path]:
    assert path.is_file(), path

run24 = json.loads(run24_path.read_text(encoding='utf-8'))
freeze24 = json.loads(freeze24_path.read_text(encoding='utf-8'))
watchlist = pd.read_csv(watchlist_path)

assert run24['status'] == 'READY_FOR_NOTEBOOK_25_IDENTITY_REVEAL', run24['status']
assert run24['identity_access'] is False
assert freeze24['status'] == 'IDENTITY_BLIND_RANKING_FROZEN', freeze24['status']
assert freeze24['identity_blind'] is True
assert len(watchlist) == 8 and watchlist['target_id'].nunique() == 8

display(pd.Series({
    'Notebook 24 status': run24['status'],
    'Notebook 24 identity access': run24['identity_access'],
    'Frozen feature predictions': freeze24['n_features'],
    'Frozen case-feature groups': freeze24['n_case_feature_groups'],
    'Published Watch-List targets': watchlist['target_id'].nunique(),
}, name='value').to_frame())
display(watchlist[['compound_name_published', 'watchlist_scope', 'source_statement']])

### 评价口径

- **全部论文CEC**：衡量公开参考谱与真实水样中的已知污染物覆盖，不代表PBT阳性集合。
- **8个Watch‑List CEC**：衡量法规/监测关注物的排序捕获，也不等同于PBT真值。
- **冻结PBT危险性排序**是主科学输出；出现度辅助排序只用于部署优先级分析，不能回写或替换24号冻结结果。
- 只报告Top 5%、10%、20%的捕获和富集；不对不完整注释计算常规precision、AUROC或“阴性”性能。

## 2. 准备公开实测参考谱与案例峰表

筛选规则为前体质量误差≤20 ppm、参考谱至少6个峰；为避免参考谱极多的化合物支配搜索，每个目标/离子模式/加合物/碰撞能/来源层最多保留2张参考谱。此阶段主要使用CPU、内存和磁盘，不需要GPU。

In [ ]:
if RUN_PREPARE:
    run_stage('prepare')

prepare_path = MANIFEST_DIR / '25_identity_reference_preparation_manifest.json'
assert prepare_path.is_file(), '未找到prepare结果；请将RUN_PREPARE设为True后运行本单元。'
prepare = json.loads(prepare_path.read_text(encoding='utf-8'))
assert prepare['status'] == 'PASS', prepare
display(pd.Series(prepare['counts'], name='count').to_frame())
display(pd.Series(prepare['matching_rules'], name='value').to_frame())

## 3. 严格谱库匹配

主匹配规则：前体误差≤20 ppm、碎片容差0.02 Da、modified cosine≥0.70、匹配峰数≥6；高置信子集为cosine≥0.85且匹配峰数≥8。若第二候选目标与最佳目标分差≤0.05，则标记为歧义，不进入非歧义已知污染物评价。

In [ ]:
if RUN_MATCH:
    run_stage('match')

match_path = MANIFEST_DIR / '25_library_match_manifest.json'
assert match_path.is_file(), '未找到match结果；请先完成prepare并将RUN_MATCH设为True。'
match = json.loads(match_path.read_text(encoding='utf-8'))
assert match['status'] == 'PASS', match
display(pd.Series(match['counts'], name='count').to_frame())
display(pd.Series(match['rules'], name='value').to_frame())
print('Identity claim:', match['identity_claim'])

## 4. 揭盲合并与Top-k评价

把非歧义Level 2a命中合并回24号冻结的case-feature排序。四种排序在相同、具有完整主危险性得分且不属于低MS2深度专属组的评价宇宙中比较，避免分母不一致。

In [ ]:
if RUN_EVALUATE:
    run_stage('evaluate')

run25_path = MANIFEST_DIR / '25_identity_reveal_run_manifest.json'
qc_path = MANIFEST_DIR / '25_identity_reveal_qc_checks.csv'
assert run25_path.is_file(), '未找到evaluate结果；请先完成前两阶段并将RUN_EVALUATE设为True。'
run25 = json.loads(run25_path.read_text(encoding='utf-8'))
qc = pd.read_csv(qc_path)
assert run25['status'] == 'READY_FOR_NOTEBOOK_26_PORTUGAL_CASE_INTERPRETATION', run25
assert run25['qc_all_required_pass'] is True
assert qc['passed'].astype(bool).all()

display(pd.Series(run25['counts'], name='count').to_frame())
display(qc)

## 5. 已知污染物恢复结果

下表首先展示8个Watch‑List CEC的恢复状态和最佳谱图证据，再展示全部论文CEC中按冻结PBT危险性排序最靠前的非歧义命中。注意：谱库恢复说明“该已知物在数据中有相符MS/MS证据”，不等于其P/B/T标签已被实验确认。

In [ ]:
recovery = pd.read_csv(OUT / '25_known_target_recovery.csv')
watch_recovery = recovery.loc[recovery['published_watchlist_cec'].astype(bool), [
    'compound_name_published', 'recovered', 'recovered_case_group_n',
    'best_modified_cosine', 'best_matched_peaks', 'hazard_rank',
    'occurrence_assisted_rank', 'frequency_baseline_rank'
]].sort_values(['recovered', 'occurrence_assisted_rank'], ascending=[False, True])
display(watch_recovery.reset_index(drop=True))

known_top = recovery.loc[recovery['recovered'].astype(bool), [
    'compound_name_published', 'main_use', 'compound_class',
    'published_detection_percent_max', 'best_modified_cosine', 'best_matched_peaks',
    'hazard_rank', 'occurrence_assisted_rank', 'frequency_baseline_rank'
]].sort_values('hazard_rank').head(25)
display(known_top.reset_index(drop=True))

## 6. Top 5%/10%/20%捕获与富集

`enrichment_fold > 1`表示该排序前部比整个共同评价宇宙更富集已知命中。全部CEC集合天然偏向高检出污染物，因此频率/丰度或出现度辅助排序通常占优；这不能用来否定PBT模型，也不能作为PBT准确率。Watch‑List结果回答的是法规关注物能否被前排捕获。

In [ ]:
enrichment = pd.read_csv(OUT / '25_top_fraction_enrichment.csv')
for scope in enrichment['label_scope'].drop_duplicates():
    print('\n', scope)
    table = enrichment.loc[enrichment['label_scope'].eq(scope), [
        'method', 'top_fraction', 'known_hit_group_n',
        'known_hit_density', 'background_known_hit_prevalence', 'enrichment_fold'
    ]].copy()
    table['top_fraction'] = (100 * table['top_fraction']).round().astype(int).astype(str) + '%'
    display(table.reset_index(drop=True))

In [ ]:
display(Image(filename=str(OUT / 'figures' / '25_known_pollutant_enrichment.png')))
display(Image(filename=str(OUT / 'figures' / '25_model_vs_library_coverage.png')))

## 7. 结果解释与下一步边界

通过标准是最终状态为 `READY_FOR_NOTEBOOK_26_PORTUGAL_CASE_INTERPRETATION` 且全部必需QC为True。模型可为全部79,867个合格MS/MS特征给出危险性预测，而严格公开谱库只能给一小部分样品特征提供已知身份；这正是“直接谱图危险性筛查”相对“必须先完成结构鉴定”的覆盖优势。

论文表述必须保留以下限制：公开谱库和论文注释不完整；没有明确空白/QC原始文件，因此不声称完成空白扣除；没有同批次保留时间标准，因此仅报告Level 2a；检出频率和峰面积不等于暴露浓度。下一步26号应做站点/批次模式、阈值敏感性和可发表图表整合，不能回头根据揭盲结果重新选模型或改变24号主排序。

In [ ]:
print('Notebook 25 final status:', run25['status'])
print('QC all required pass:', run25['qc_all_required_pass'])
print('Recovered published CEC targets:',
      f"{run25['counts']['recovered_unique_targets']}/{run25['counts']['published_unique_targets']}")
print('Recovered article-defined Watch-List targets:',
      f"{run25['counts']['recovered_watchlist_targets']}/{run25['counts']['published_watchlist_targets']}")
print('Next step:', run25['next_step'])